# Modern UEC-operation-v1

Bounded operation handling, not vulnerability proof. Historical PR #10 is preserved separately. No independent human reviewer was available.

In [1]:
from pprint import pprint
from collections import Counter
from ml.modern.dataset import DATA, RESULTS, read_json, validate_labels, validate_split
from ml.modern.experiment import prepare, train_models, evaluate


# Dataset audit

Unchanged pinned files compiled through the actual solc 0.8.20 backend. Source/license/provenance details and all exclusions are in the manifest.

In [2]:
inventory=read_json(DATA/'inventory.json')
pprint({'sources':len(inventory), 'compile_status':dict(Counter(r['compilation']['status'] for r in inventory)), 'families':len({r['group'] for r in inventory})})

{'compile_status': {'compilation_error': 57, 'compiled': 44},
 'families': 11,
 'sources': 101}


# Label validation

Positive means discarded/unhandled success. Require/assert or effective failure branches are negative. Unknown and unsupported are excluded. High-level methods named call are outside the target.

In [3]:
labels=read_json(DATA/'labels.json')
candidates=read_json(RESULTS/'candidates.json')
validate_labels(labels,inventory,candidates)
pprint(dict(Counter(r['label'] for r in labels)))

{'negative': 36, 'positive': 18, 'unknown': 7, 'unsupported': 17}


# Frozen family split

The split and input hashes were committed before model fitting. Controlled cases are training-only. Related SBE/Ethernaut tutorials share one family. The tiny eligible test set cannot pass the 20-family gate.

In [4]:
split=read_json(DATA/'split.json')
validate_split(inventory,split)
pprint({part:len({s['family'] for s in split.values() if s['partition']==part}) for part in ['train','validation','test']})

{'test': 3, 'train': 5, 'validation': 3}


# Feature extraction

Reuse the unchanged typed-AST schema. No rule outputs, dataset categories, source names or test-set statistics enter model inputs. Verify parity against actual backend compilation.

In [5]:
rows=prepare()
pprint({part:sum(r['eligible'] and r['partition']==part for r in rows) for part in ['train','validation','test']})

{'test': 4, 'train': 16, 'validation': 33}


# Baseline training and gradient boosting

Fit dummy, fixed logistic with training-only scaling, and four HGB configurations. Validation chooses thresholds and HGB settings; test is not an argument to selection.

In [6]:
fitted,selection=train_models(rows)
pprint(selection)

{'preferred': 'gradient_boosting',
 'rationale': 'HGB requires at least 0.02 validation macro-F1 advantage; '
              'otherwise prefer logistic.',
 'trials': [{'model': 'dummy',
             'parameters': {},
             'threshold': 0.5,
             'validation_macro_f1': 0.1951219512195122,
             'validation_positive_precision': 0.24242424242424243},
            {'model': 'logistic',
             'parameters': {},
             'threshold': 0.5,
             'validation_macro_f1': 0.9093406593406593,
             'validation_positive_precision': 1.0},
            {'model': 'gradient_boosting',
             'parameters': {'max_leaf_nodes': 3, 'min_samples_leaf': 2},
             'threshold': 0.5,
             'validation_macro_f1': 0.9568627450980391,
             'validation_positive_precision': 1.0},
            {'model': 'gradient_boosting',
             'parameters': {'max_leaf_nodes': 3, 'min_samples_leaf': 5},
             'threshold': 0.5,
             'validatio

# Validation selection

Use the frozen macro-F1/precision/tie-break rules. HGB needs at least 0.02 validation macro-F1 advantage over logistic. Do not refit on validation.

In [7]:
pprint({'preferred':selection['preferred'], 'thresholds':{name:item['threshold'] for name,item in fitted.items()}})

{'preferred': 'gradient_boosting',
 'thresholds': {'dummy': 0.5, 'gradient_boosting': 0.5, 'logistic': 0.5}}


# Final test evaluation

This cell reproduces the single fixed final evaluation and asserts equality with committed results. It never searches settings. C++ unsupported coverage is an abstention.

In [8]:
result=evaluate(rows,fitted,selection,reproduce=True)
pprint(result['counts'])
pprint({name: {'matrix':item['confusion_matrix'],'macro_f1':item['macro_f1'],'family_bootstrap':item['family_bootstrap']} for name,item in result['models'].items()})
pprint(result['rule_completed'])

{'test': {'classes': {'negative': 2, 'positive': 2},
          'eligible_operations': 4,
          'exclusions': {'label: unsupported': 3},
          'families': 1,
          'kinds': {'benchmark-derived': 4},
          'reviewed_operations': 7},
 'train': {'classes': {'negative': 8, 'positive': 8},
           'eligible_operations': 16,
           'exclusions': {'Assignment/return data flow outside feature domain': 1,
                          'label: unknown': 2,
                          'label: unsupported': 5},
           'families': 2,
           'kinds': {'benchmark-derived': 1, 'controlled': 15},
           'reviewed_operations': 24},
 'validation': {'classes': {'negative': 25, 'positive': 8},
                'eligible_operations': 33,
                'exclusions': {'label: unknown': 5, 'label: unsupported': 9},
                'families': 2,
                'kinds': {'benchmark-derived': 33},
                'reviewed_operations': 47}}
{'dummy': {'family_bootstrap': {'families'

# Error analysis and strata

Exact false positives/negatives retain source spans and reviewed handling rationales. Report real-world and controlled test denominators as zero when absent. One test family cannot support a family-bootstrap interval.

In [9]:
pprint(read_json(RESULTS/'errors.json'))
pprint({name:{'by_kind':item['by_kind'],'by_pattern':item['by_pattern']} for name,item in result['models'].items()})

[{'error': 'false_positive',
  'expected': 'negative',
  'explanation': 'The prediction contradicts the reviewed handling shown in '
                 'the rationale. These syntactic features do not prove '
                 'result-handling semantics; no feature/threshold adjustment '
                 'follows this test error.',
  'family': 'slither:tests/e2e/detectors/test_data/low-level-calls/0.4.25/low_level_calls.sol',
  'features': {'argument_count': 1,
               'assigned_result': 1,
               'condition_ancestor': 0,
               'delegatecall': 0,
               'function_statement_count': 2,
               'guard_ancestor': 0,
               'later_result_references': 1,
               'standalone_expression': 0,
               'staticcall': 0,
               'unary_ancestor': 0,
               'value_option': 1},
  'function': 'good',
  'model': 'dummy',
  'rationale': "require(ret) checks this operation's success.",
  'score': 0.5,
  'source_id': 'slither:tests/e2

# Integration decision

No model artifact, ML API fields or frontend panel is justified when the gate fails. Perfect classification of four benchmark operations is not real-world accuracy.

In [10]:
pprint(result['integration'])
pprint(result['limitations'])

{'candidate_selected_on_validation': 'gradient_boosting',
 'checks': {'at_least_10_per_class': False,
            'at_least_20_test_families': False,
            'at_least_two_real_world_test_families': False,
            'backend_feature_parity': True,
            'beats_dummy_macro_f1': True,
            'binary_test_coverage_at_least_80_percent': True,
            'false_positive_rate_at_most_5_percent': True,
            'hgb_beats_logistic_if_selected': False,
            'incremental_value_on_completed_rule_cases': False,
            'paired_advantage_over_dummy': False,
            'paired_advantage_over_logistic': False,
            'positive_precision_lower_bound_at_least_80_percent': False},
 'failed_checks': ['at_least_20_test_families',
                   'at_least_10_per_class',
                   'at_least_two_real_world_test_families',
                   'hgb_beats_logistic_if_selected',
                   'positive_precision_lower_bound_at_least_80_percent',
           